# Hyperion Economy — lernende Agenten in Juno

Drei lokale Agenten beobachten echte Hyperion-Marktdaten: Reliktpreis, Stabilität, Time Debt, Ereignisse und Hyperion-Bestand. Sie handeln eine kleine Overlay-Position und lernen mit SARSA, Dyna-Q oder einem Bandit-Modell.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from learning_agents import (
    train_agents, evaluate_agents, evaluation_summary, load_agent_memory
)

MEMORY_FILE = Path('agent_memory.json')
EPISODES = 30
YEARS = 20
SEED = 7
print('Lokales Agentenmodul geladen.')

## 1. Training

Für eine schnelle Präsentation reichen 10 Episoden. 30–50 Episoden machen das Verhalten stabiler. Das Gedächtnis wird als versionierte JSON-Datei gespeichert.

In [ ]:
result = train_agents(
    episodes=EPISODES,
    years=YEARS,
    seed=SEED,
    memory_path=str(MEMORY_FILE),
)
history = pd.DataFrame(result['history'])
print(f'{len(result["agents"])} Agenten trainiert.')
print(f'Gedächtnis: {MEMORY_FILE.resolve()}')
display(history.groupby('agent').tail(1)[[
    'agent', 'algorithm', 'final_value', 'benchmark_value',
    'excess_return', 'trades', 'q_states', 'epsilon'
]])

## 2. Lernkurven

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for agent, frame in history.groupby('agent'):
    axes[0].plot(
        frame['episode'],
        frame['total_reward'].rolling(5, min_periods=1).mean(),
        label=agent,
    )
    axes[1].plot(frame['episode'], frame['excess_return'], label=agent)
axes[0].set_title('Gleitender Lernfortschritt')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Reward')
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('Mehr-/Minderertrag gegenüber Buy-and-Hold')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Excess Return')
for axis in axes:
    axis.legend()
    axis.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 3. Unabhängige Bewertung

Die Bewertung verwendet neue Hyperion-Jahre und schaltet Exploration aus. Zusätzlich werden Buy-and-Hold, Streuung, Liquiditätsstress und Insolvenzen ausgewiesen.

In [ ]:
evaluation = evaluate_agents(
    result['agents'], episodes=5, years=YEARS, seed=SEED + 1000
)
evaluation_frame = pd.DataFrame(evaluation)
summary = pd.DataFrame(evaluation_summary(evaluation))
display(summary.round(2))

In [ ]:
fig, axis = plt.subplots(figsize=(10, 4))
evaluation_frame.boxplot(
    column='final_value', by='agent', ax=axis, grid=False
)
axis.set_title('Endwert in neuen Hyperion-Marktverläufen')
axis.set_xlabel('Agentenrolle')
axis.set_ylabel('Portfolio-Endwert')
plt.suptitle('')
plt.tight_layout()
plt.show()

## 4. Gedächtnis laden und fortsetzen

Die folgende Zelle lädt das Gedächtnis. Für ein weiteres Training kann die auskommentierte Zeile aktiviert werden.

In [ ]:
restored = load_agent_memory(str(MEMORY_FILE), seed=SEED)
memory_view = pd.DataFrame({
    'agent': [agent.name for agent in restored],
    'algorithm': [agent.algorithm for agent in restored],
    'q_states': [len(agent.q_table) for agent in restored],
    'bandit_actions': [sum(agent.bandit_counts.values()) for agent in restored],
    'epsilon': [agent.epsilon for agent in restored],
})
display(memory_view)
# continued = train_agents(episodes=10, years=YEARS, seed=SEED + 200, agents=restored, memory_path=str(MEMORY_FILE))

## 5. Optional: interaktiver Kurztest

In [ ]:
try:
    import ipywidgets as widgets
    def quick_training(episodes=10, years=12):
        quick = train_agents(episodes=episodes, years=years, seed=SEED, memory_path=None)
        display(pd.DataFrame(evaluation_summary(
            evaluate_agents(quick['agents'], episodes=3, years=years, seed=SEED + 500)
        )).round(2))
    display(widgets.interact(
        quick_training,
        episodes=widgets.IntSlider(value=10, min=5, max=60, step=5),
        years=widgets.IntSlider(value=12, min=5, max=30, step=1),
    ))
except ImportError:
    print('ipywidgets ist optional. Die festen Notebook-Zellen funktionieren trotzdem.')

### Demo-Erzählung

- **Profit-Scout / SARSA** lernt vorsichtig aus der tatsächlich gewählten Folgeaktion.
- **Reserve-Keeper / Dyna-Q** nutzt zusätzlich gespeicherte Übergänge und wird für Instabilität, Time Debt und Ereignisse stärker belastet.
- **Hyperion-Speculator / Bandit** lernt die durchschnittliche Wirkung einzelner Aktionen und dient als bewusst einfacherer Vergleich.

Alle drei Agenten sehen dieselben Hyperion-Marktsnapshots. Die Agentenbörse bleibt als Overlay getrennt von den Weltkonten der Wirtschaftssimulation.